# Temporal progression with partially unfrozen CT-CLIP

This notebook uses the full labeled longitudinal CT-RATE cohort while adapting only the
last CTViT image-encoder block. It uses only report-grounded, explicit temporal labels;
cross-domain pairs and scans without usable findings are excluded.

Expected cohort:

- 3,172 training pairs
- 555 tuning/validation pairs
- 233 held-out test pairs
- 6,590 unique prior/current CT volumes

## What is trainable

- final (`4th`) temporal CTViT transformer block;
- final temporal CTViT normalization;
- finding-conditioned Difference Transformer and classification/magnitude heads.

Everything else stays frozen, including the spatial CTViT blocks, first three temporal
blocks, vector quantizer, 294,912→512 visual projection, and complete text tower.

## Why there are two stages

Raw CTs are downloaded from gated CT-RATE only once. The notebook caches the output of the
frozen image prefix immediately before the trainable final block. Each training step then
recomputes a fresh 512-d CT-CLIP embedding through the trainable final block. The frozen
boundary is cached in BF16 (the same precision used for training), avoiding repeated
download/preprocessing of two large `480×480×240` volumes every epoch.

The prefix cache is about 14 MiB/volume in BF16 (~90 GiB for 6,590 scans). Use a GPU with
BF16 support (A100 or L4 recommended) and at least 105 GiB of Google Drive free space.

In [ ]:
!nvidia-smi
import os, platform, sys, torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU.'
assert torch.cuda.is_bf16_supported(), 'A BF16-capable GPU (A100/L4) is required.'
print('Python:', platform.python_version())
print('Torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0), '| BF16:', torch.cuda.is_bf16_supported())

## 1. Install pinned CT-CLIP source

The image architecture is pinned so the final-block path cannot silently drift.

In [ ]:
%cd /content

import shutil
import subprocess
from pathlib import Path

CTCLIP_COMMIT = "a2a155c601987820433c01db69b64d701d3d229d"
CTCLIP_DIR = Path("/content/CT-CLIP")
ARCHIVE = Path("/content/ctclip_source.tar.gz")
EXTRACTED = Path(f"/content/CT-CLIP-{CTCLIP_COMMIT}")

shutil.rmtree(CTCLIP_DIR, ignore_errors=True)
shutil.rmtree(EXTRACTED, ignore_errors=True)
ARCHIVE.unlink(missing_ok=True)

subprocess.run(
    [
        "curl",
        "--fail",
        "--location",
        "--retry", "5",
        "--retry-all-errors",
        "--output", str(ARCHIVE),
        "https://codeload.github.com/ibrahimethemhamamci/"
        f"CT-CLIP/tar.gz/{CTCLIP_COMMIT}",
    ],
    check=True,
)

subprocess.run(["tar", "-xzf", str(ARCHIVE), "-C", "/content"], check=True)

assert EXTRACTED.is_dir(), list(Path("/content").glob("CT-CLIP-*"))
EXTRACTED.rename(CTCLIP_DIR)
ARCHIVE.unlink()

print("Pinned CT-CLIP source installed:", CTCLIP_DIR)


## 2. Mount Drive and configure the experiment

Before running the next cell, place this exact file in `MyDrive/3dCT/manifests/`:

1. `medgemma_labels_v6_synthetic_direction.jsonl` from the project root.

It defines the exact v6 labels and standardized direction text targets. This report-derived
local artifact is not published by this notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import gc, hashlib, json, math, random, shutil, time
from collections import Counter
from pathlib import Path
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import balanced_accuracy_score, f1_score
from tqdm.auto import tqdm

DRIVE_ROOT = Path('/content/drive/MyDrive/3dCT')
MANIFEST_DIR = DRIVE_ROOT / 'manifests'
LABEL_JSONL = MANIFEST_DIR / 'medgemma_labels_v6_synthetic_direction.jsonl'
WEIGHTS_DIR = DRIVE_ROOT / 'ctclip_weights'
RUN_ROOT = DRIVE_ROOT / 'ctclip_lastblock_temporal'
PREFIX_DIR = RUN_ROOT / 'prefix_cache'
TEXT_CACHE = RUN_ROOT / 'text_targets.pt'
CHECKPOINT_DIR = RUN_ROOT / 'checkpoints'
RESULT_DIR = RUN_ROOT / 'results'
TMP = Path('/content/_ctclip_temporal_volume')
LOCAL_PREFIX_DIR = Path('/content/ctclip_prefix_cache')

for directory in [MANIFEST_DIR, WEIGHTS_DIR, PREFIX_DIR, CHECKPOINT_DIR, RESULT_DIR, TMP]:
    directory.mkdir(parents=True, exist_ok=True)

if not LABEL_JSONL.is_file():
    from google.colab import files
    print('Select medgemma_labels_v6_synthetic_direction.jsonl from your local project.')
    uploaded = files.upload()
    for name, content in uploaded.items():
        if name == 'medgemma_labels_v6_synthetic_direction.jsonl':
            (MANIFEST_DIR / name).write_bytes(content)
assert LABEL_JSONL.is_file(), f'Missing {LABEL_JSONL}'

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while block := handle.read(chunk_size):
            digest.update(block)
    return digest.hexdigest()

LABEL_SHA256 = sha256_file(LABEL_JSONL)
assert LABEL_SHA256 == '04ca9952eeb365a3e9bebd7e96f58a82af22f1bf78d06e25eaabd90261457e66', LABEL_SHA256
print('Temporal label-manifest hash verified.')

SEED = 2026
TUNE_FRAC = 0.15
COHORT_ID = 'v6_synthetic_direction_report_explicit_grounded_v1'
EPOCHS = 10
PATIENCE = 3
IMAGE_LR = 1e-6
TEMPORAL_LR = 1e-4
WEIGHT_DECAY = 1e-2
GRAD_ACCUM_PAIRS = 8
LAMBDA_CE = 1.0
LAMBDA_MAG = 0.5
LAMBDA_TEXT = 0.5
MAX_GRAD_NORM = 1.0
STAGE_PREFIX_TO_LOCAL = True
DEVICE = torch.device('cuda')

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('Run root:', RUN_ROOT)

## 3. Authenticate and download official CT-CLIP weights

In [ ]:
from huggingface_hub import get_token, hf_hub_download, login

login()  # READ token after accepting CT-RATE terms
raw_token = os.environ.get('HF_TOKEN') or get_token()
assert raw_token, 'No Hugging Face token available.'
HF_TOKEN = ''.join(str(raw_token).split())
assert HF_TOKEN.startswith('hf_') and HF_TOKEN.isascii()
os.environ['HF_TOKEN'] = HF_TOKEN

CTRATE_REPO = 'ibrahimhamamci/CT-RATE'
CTRATE_REVISION = 'deeca4d89e9f978d4d1bccd88a55071ddbb146bb'
CTCLIP_WEIGHTS_REMOTE = 'models/CT-CLIP-Related/CT-CLIP_v2.pt'
weights_path = Path(hf_hub_download(
    CTRATE_REPO, CTCLIP_WEIGHTS_REMOTE, repo_type='dataset', revision=CTRATE_REVISION,
    token=HF_TOKEN, local_dir=str(WEIGHTS_DIR)))
print('Weights:', weights_path, '| GiB:', round(weights_path.stat().st_size / 1024**3, 2))

## 4. Reconstruct the full longitudinal temporal cohort

Only v6 findings with a verified, report-grounded temporal direction (`worsened`, `stable`,
or `improved`) are used. The classification labels originate in real report text; the frozen
text-alignment targets are the deterministic standardized `training_temporal_sentence` fields.
Same-domain `train_*` pairs are assigned to train/tune by a deterministic patient-level hash;
same-domain `valid_*` pairs are held out for test. Cross-domain pairs are excluded.

In [ ]:
CLASSES = ['worsened', 'stable', 'improved']
C2I = {name: index for index, name in enumerate(CLASSES)}
FINDINGS = [
    'Medical material', 'Arterial wall calcification', 'Cardiomegaly',
    'Pericardial effusion', 'Coronary artery wall calcification', 'Hiatal hernia',
    'Lymphadenopathy', 'Emphysema', 'Atelectasis', 'Lung nodule', 'Lung opacity',
    'Pulmonary fibrotic sequela', 'Pleural effusion', 'Mosaic attenuation pattern',
    'Peribronchial thickening', 'Consolidation', 'Bronchiectasis',
    'Interlobular septal thickening',
]
F2I = {name: index for index, name in enumerate(FINDINGS)}

label_rows = [json.loads(line) for line in LABEL_JSONL.open(encoding='utf-8') if line.strip()]

def split_train_patient(patient):
    raw = f'{SEED}|{patient}'.encode('utf-8')
    u = int.from_bytes(hashlib.sha256(raw).digest()[:8], 'big') / 2**64
    return 'tune' if u < TUNE_FRAC else 'train'

records = {'train': [], 'tune': [], 'test': []}
skipped = Counter()
for label in label_rows:
    if not label.get('parse_ok'):
        skipped['parse_failed'] += 1; continue
    prior_volume, curr_volume = label['prior_volume'], label['curr_volume']
    if prior_volume.startswith('train_') and curr_volume.startswith('train_'):
        split = split_train_patient(label['patient'])
    elif prior_volume.startswith('valid_') and curr_volume.startswith('valid_'):
        split = 'test'
    else:
        skipped['cross_domain_or_unknown'] += 1; continue
    findings = []
    for finding in label.get('findings', []):
        if not finding.get('progression_eligible'):
            continue
        if finding.get('original_label_source') != 'report_explicit':
            continue
        if finding.get('label_source') != 'report_explicit' or not finding.get('report_grounded'):
            continue
        direction = finding.get('direction')
        name = finding.get('finding')
        if direction not in C2I or name not in F2I:
            continue
        training_text = (finding.get('training_temporal_sentence') or '').strip()
        assert training_text, f'Missing synthetic direction target: {label["curr_volume"]}, {name}'
        findings.append({'fid': F2I[name], 'finding': name, 'y': C2I[direction],
                         'direction': direction, 'text': training_text})
    if not findings:
        skipped['no_usable_report_grounded_finding'] += 1; continue
    records[split].append({
        'patient': label['patient'], 'prior_volume': prior_volume,
        'curr_volume': curr_volume, 'delta_days': int(label.get('delta_days') or 0),
        'findings': findings,
    })

assert {split: len(rows) for split, rows in records.items()} == {
    'train': 3172, 'tune': 555, 'test': 233}
patients = {split: {row['patient'] for row in rows} for split, rows in records.items()}
assert patients['train'].isdisjoint(patients['tune'])
assert patients['train'].isdisjoint(patients['test'])
assert patients['tune'].isdisjoint(patients['test'])

target_volumes = sorted({row[key] for split in records for row in records[split]
                         for key in ('prior_volume', 'curr_volume')})
assert len(target_volumes) == 6590, len(target_volumes)
finding_counts = {split: sum(len(row['findings']) for row in rows)
                  for split, rows in records.items()}
assert finding_counts == {'train': 10603, 'tune': 1882, 'test': 795}, finding_counts
print('Pairs:', {split: len(rows) for split, rows in records.items()})
print('Finding examples:', finding_counts)
print('Unique temporal volumes:', len(target_volumes))
print('Skipped labels:', dict(skipped))

## 5. CT-CLIP preprocessing and gated volume download

In [ ]:
import nibabel as nib
from scipy.ndimage import zoom

HU_MIN, HU_MAX = -1000.0, 200.0
TARGET_SPACING = (0.75, 0.75, 1.5)
TARGET_SHAPE_HWD = (480, 480, 240)
PREPROCESSING_ID = 'ctclip_hu-1000_200_spacing075_075_15_crop480_480_240_v1'

def resample_to_spacing(volume, spacing_xyz, target_xyz):
    sx, sy, sz = spacing_xyz; tx, ty, tz = target_xyz
    factors = (sy / ty, sx / tx, sz / tz)
    return volume if all(abs(value - 1.0) < 1e-3 for value in factors) else zoom(
        volume, factors, order=1)

def center_crop_pad(volume, target_hwd, pad_value):
    output = np.full(target_hwd, pad_value, dtype=volume.dtype)
    source_slices, destination_slices = [], []
    for source, target in zip(volume.shape, target_hwd):
        if source >= target:
            start = (source - target) // 2
            source_slices.append(slice(start, start + target)); destination_slices.append(slice(0, target))
        else:
            start = (target - source) // 2
            source_slices.append(slice(0, source)); destination_slices.append(slice(start, start + source))
    output[tuple(destination_slices)] = volume[tuple(source_slices)]
    return output

def preprocess_ct(path):
    image = nib.load(path)
    volume = image.get_fdata(dtype=np.float32, caching='unchanged')
    spacing = image.header.get_zooms()[:3]
    volume = np.clip(volume, HU_MIN, HU_MAX)
    volume = ((volume - HU_MIN) / (HU_MAX - HU_MIN)) * 2.0 - 1.0
    volume = resample_to_spacing(volume, spacing, TARGET_SPACING)
    volume = center_crop_pad(volume, TARGET_SHAPE_HWD, -1.0)
    volume = np.ascontiguousarray(volume.transpose(2, 0, 1))
    tensor = torch.from_numpy(volume).float().unsqueeze(0).unsqueeze(0)
    del volume, image
    return tensor

def remote_candidates(volume):
    base = volume.removesuffix('.nii.gz').removesuffix('.nii')
    split, patient_id, scan = base.split('_')[:3]
    patient = f'{split}_{patient_id}'; scan_folder = f'{split}_{patient_id}_{scan}'
    return [f'dataset/{folder}/{patient}/{scan_folder}/{volume}'
            for folder in (f'{split}_fixed', split)]

def reset_tmp():
    shutil.rmtree(TMP, ignore_errors=True); TMP.mkdir(parents=True, exist_ok=True)

def download_volume(volume):
    reset_tmp(); errors = []
    for remote in remote_candidates(volume):
        try:
            return Path(hf_hub_download(
                CTRATE_REPO, remote, repo_type='dataset', revision=CTRATE_REVISION,
                token=HF_TOKEN, local_dir=str(TMP)))
        except Exception as exc:
            errors.append(f'{remote}: {type(exc).__name__}: {exc}')
    raise RuntimeError('\n'.join(errors))

In [ ]:
%cd /content

import subprocess
import sys
from pathlib import Path

ROOT = Path("/content/CT-CLIP")

assert (ROOT / "CT_CLIP" / "ct_clip").is_dir()
assert (ROOT / "transformer_maskgit" / "transformer_maskgit").is_dir()

# Install only dependencies actually needed by the CT-CLIP model code.
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "beartype",
        "einops>=0.6",
        "ftfy",
        "regex",
        "ema-pytorch>=0.2.2",
        "torchtyping",
        "vector-quantize-pytorch==1.1.2",
        "sentencepiece",
        "nibabel",
        "openpyxl",
        "accelerate",
    ],
    check=True,
)

# Register local source packages WITHOUT asking pip to fetch the unnecessary
# GitHub-only ImageNetV2 dependency.
for path in [str(ROOT / "CT_CLIP"), str(ROOT / "transformer_maskgit")]:
    if path not in sys.path:
        sys.path.insert(0, path)

# Verify the required imports.
import ct_clip
import transformer_maskgit
from ct_clip import CTCLIP
from transformer_maskgit import CTViT

print("CTCLIP import works:", CTCLIP)
print("CTViT import works:", CTViT)
print("ct_clip source:", ct_clip.__file__)


## 6. Build CT-CLIP and load official weights

In [ ]:
from transformers import BertModel
from ct_clip import CTCLIP
from transformer_maskgit import CTViT

text_encoder = BertModel.from_pretrained('microsoft/BiomedVLP-CXR-BERT-specialized')
image_encoder = CTViT(
    dim=512, codebook_size=8192, image_size=480, patch_size=20,
    temporal_patch_size=10, spatial_depth=4, temporal_depth=4,
    dim_head=32, heads=8,
)
clip = CTCLIP(
    image_encoder=image_encoder, text_encoder=text_encoder,
    dim_image=294912, dim_text=768, dim_latent=512,
    extra_latent_projection=False, use_mlm=False,
    downsample_image_embeds=False, use_all_token_embeds=False,
)
state = torch.load(weights_path, map_location='cpu', weights_only=False)
if isinstance(state, dict) and 'model' in state and isinstance(state['model'], dict):
    state = state['model']
missing, unexpected = clip.load_state_dict(state, strict=False)
benign = {'text_transformer.embeddings.position_ids'}
real_unexpected = [key for key in unexpected if key not in benign]
assert not missing and not real_unexpected, (missing[:20], real_unexpected[:20])
del state, text_encoder, image_encoder

for parameter in clip.parameters():
    parameter.requires_grad_(False)
clip.eval()

visual = clip.visual_transformer
assert len(visual.enc_spatial_transformer.layers) == 4
assert len(visual.enc_temporal_transformer.layers) == 4
FINAL_VISUAL_BLOCK = visual.enc_temporal_transformer.layers[-1]
FINAL_VISUAL_NORM = visual.enc_temporal_transformer.norm_out
print('Final trainable block:', FINAL_VISUAL_BLOCK)
print('CT-CLIP loaded cleanly.')

## 7. Cache the frozen visual prefix for the 6,590 temporal scans

This is resumable. Each payload stores the tensor immediately before the final temporal
CTViT block: `(576, 24, 512)` in BF16. Raw NIfTIs are removed after every volume.

In [ ]:
from einops import rearrange

PREFIX_SHAPE = (576, 24, 512)
PREFIX_ID = f'{CTCLIP_COMMIT}_{PREPROCESSING_ID}_before_temporal_layer4_bf16_v1'

def apply_transformer_layer(layer, tokens, video_shape):
    peg, self_attention, cross_attention, feed_forward = layer
    if peg is not None:
        tokens = peg(tokens, shape=video_shape) + tokens
    tokens = self_attention(tokens) + tokens
    # CTViT encoder layers do not have cross-attention.
    assert cross_attention is None
    return feed_forward(tokens) + tokens

# Verify our split execution exactly matches the official temporal Transformer before
# using it on medical data.
visual.to(DEVICE)
visual.eval()
with torch.inference_mode():
    probe_shape = (1, 2, 2, 2)
    probe = torch.randn(4, 2, 512, device=DEVICE)
    direct = visual.enc_temporal_transformer(probe.clone(), video_shape=probe_shape)
    split_probe = probe.clone()
    for layer in visual.enc_temporal_transformer.layers:
        split_probe = apply_transformer_layer(layer, split_probe, probe_shape)
    split_probe = visual.enc_temporal_transformer.norm_out(split_probe)
    assert torch.allclose(direct, split_probe, atol=1e-5, rtol=1e-5)
print('Split-forward transformer equivalence passed.')

@torch.inference_mode()
def extract_frozen_prefix(volume_tensor):
    visual.eval()
    tensor = volume_tensor.to(DEVICE, non_blocking=True)
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        tokens = visual.to_patch_emb(tensor)
        batch, temporal, height, width, dim = tokens.shape
        video_shape = (batch, temporal, height, width)
        tokens = rearrange(tokens, 'b t h w d -> (b t) (h w) d')
        bias = visual.spatial_rel_pos_bias(height, width, device=DEVICE)
        tokens = visual.enc_spatial_transformer(tokens, attn_bias=bias, video_shape=video_shape)
        tokens = rearrange(tokens, '(b t) (h w) d -> b t h w d',
                           b=batch, t=temporal, h=height, w=width)
        tokens = rearrange(tokens, 'b t h w d -> (b h w) t d')
        for layer in visual.enc_temporal_transformer.layers[:-1]:
            tokens = apply_transformer_layer(layer, tokens, video_shape)
        tokens = tokens.reshape(batch, height * width, temporal, dim)
    prefix = tokens[0].float().cpu().bfloat16().contiguous()
    assert tuple(prefix.shape) == PREFIX_SHAPE and torch.isfinite(prefix).all()
    return prefix

def prefix_path(volume, root=PREFIX_DIR):
    return root / f"{volume.removesuffix('.nii.gz').removesuffix('.nii')}.pt"

def valid_prefix(path):
    if not path.is_file(): return False
    try:
        payload = torch.load(path, map_location='cpu', weights_only=False)
        prefix = payload['prefix']
        return (payload.get('prefix_id') == PREFIX_ID and prefix.dtype == torch.bfloat16
                and tuple(prefix.shape) == PREFIX_SHAPE
                and bool(torch.isfinite(prefix).all()))
    except Exception:
        return False

def atomic_save(payload, destination):
    temporary = destination.with_suffix(destination.suffix + '.tmp')
    torch.save(payload, temporary); os.replace(temporary, destination)

done = failures = 0
for volume in tqdm(target_volumes, desc='Frozen-prefix cache'):
    output = prefix_path(volume)
    if valid_prefix(output):
        done += 1; continue
    output.unlink(missing_ok=True)
    downloaded = tensor = prefix = None
    try:
        downloaded = download_volume(volume)
        tensor = preprocess_ct(downloaded)
        prefix = extract_frozen_prefix(tensor)
        atomic_save({'format_version': 1, 'volume': volume, 'prefix': prefix,
                     'prefix_id': PREFIX_ID, 'created_unix': time.time()}, output)
        assert valid_prefix(output)
        done += 1
    except Exception as exc:
        failures += 1; print('PREFIX FAIL', volume, repr(exc), flush=True)
    finally:
        reset_tmp()
        del downloaded, tensor, prefix
        gc.collect(); torch.cuda.empty_cache()

print('Prefix cache:', done, '/', len(target_volumes), '| failures:', failures)
assert done == len(target_volumes) and failures == 0, 'Rerun this cell to retry missing prefixes.'

## 7b. Verify cached-prefix equivalence on a real CT

Before training, compare one image embedding from the original full CT-CLIP forward with
the cached BF16 prefix completed through the final temporal block, VQ, and frozen visual
projection. Cosine similarity must exceed 0.999.

In [ ]:
@torch.inference_mode()
def complete_prefix_frozen(prefix):
    prefix = prefix.unsqueeze(0).to(DEVICE)
    batch, spatial, temporal, dim = prefix.shape
    tokens = prefix.reshape(batch * spatial, temporal, dim)
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        tokens = apply_transformer_layer(
            visual.enc_temporal_transformer.layers[-1], tokens,
            (batch, temporal, 24, 24))
        tokens = visual.enc_temporal_transformer.norm_out(tokens)
        tokens = rearrange(tokens, '(b h w) t d -> b t h w d', b=batch, h=24, w=24)
        flat = tokens.reshape(batch, temporal * 24 * 24, dim)
        quantized, _, _ = visual.vq(flat)
        quantized = quantized.reshape(batch, temporal, 24, 24, dim)
        features = quantized.mean(dim=1).reshape(batch, -1)
        return F.normalize(clip.to_visual_latent(features).float(), dim=-1)

equivalence_volume = target_volumes[0]
downloaded = download_volume(equivalence_volume)
equivalence_tensor = preprocess_ct(downloaded).to(DEVICE)
visual.eval(); clip.to_visual_latent.to(DEVICE)
with torch.inference_mode(), torch.autocast(device_type='cuda', dtype=torch.bfloat16):
    full_tokens = visual(equivalence_tensor, return_encoded_tokens=True)
    full_features = full_tokens.mean(dim=1).reshape(1, -1)
    full_embedding = F.normalize(clip.to_visual_latent(full_features).float(), dim=-1)
cached_prefix = torch.load(prefix_path(equivalence_volume), map_location='cpu',
                           weights_only=False)['prefix']
split_embedding = complete_prefix_frozen(cached_prefix)
equivalence_cosine = float(F.cosine_similarity(full_embedding, split_embedding).item())
print('Full-vs-prefix embedding cosine:', equivalence_cosine)
assert equivalence_cosine > 0.999, equivalence_cosine
reset_tmp(); del downloaded, equivalence_tensor, full_tokens, full_features
del full_embedding, cached_prefix, split_embedding

## 8. Build frozen CT-CLIP text targets and class prototypes

The text tower remains frozen. Standardized v6 synthetic-direction targets and
finding/direction prompts are encoded once and cached to Drive. The labels themselves remain
strictly report-grounded; this controlled text only avoids report-writing-style noise.

In [ ]:
TEMPLATES = {
    'worsened': ['{f} has increased compared to the prior study',
                 '{f} has worsened since the previous exam', 'new {f}'],
    'stable': ['{f} is unchanged compared to the prior study',
               'stable {f} with no interval change', 'no significant change in {f}'],
    'improved': ['{f} has decreased compared to the prior study',
                 '{f} has improved since the previous exam', '{f} has resolved'],
}
TEXT_CACHE_ID = f'{CTCLIP_COMMIT}_{COHORT_ID}_training_text_templates_v1'

all_texts = sorted({finding['text'] for split in records for row in records[split]
                    for finding in row['findings']} |
                   {template.format(f=finding.lower()) for finding in FINDINGS
                    for class_name in CLASSES for template in TEMPLATES[class_name]})

@torch.inference_mode()
def encode_texts(texts, batch_size=64):
    tokenizer = clip.tokenizer
    outputs = []
    for start in tqdm(range(0, len(texts), batch_size), desc='Text targets'):
        encoded = tokenizer(texts[start:start + batch_size], padding='max_length',
                            truncation=True, max_length=512, return_tensors='pt').to(DEVICE)
        hidden = clip.text_transformer(
            input_ids=encoded['input_ids'], attention_mask=encoded['attention_mask'])[0][:, 0]
        outputs.append(F.normalize(clip.to_text_latent(hidden).float(), dim=-1).cpu())
    return torch.cat(outputs)

text_payload = None
if TEXT_CACHE.is_file():
    candidate = torch.load(TEXT_CACHE, map_location='cpu', weights_only=False)
    if candidate.get('text_cache_id') == TEXT_CACHE_ID and candidate.get('texts') == all_texts:
        text_payload = candidate
if text_payload is None:
    visual.cpu(); gc.collect(); torch.cuda.empty_cache()
    clip.text_transformer.to(DEVICE); clip.to_text_latent.to(DEVICE); clip.eval()
    embeddings = encode_texts(all_texts)
    lookup = {text: embeddings[index] for index, text in enumerate(all_texts)}
    prototypes = torch.empty(len(FINDINGS), len(CLASSES), 512)
    for fid, finding in enumerate(FINDINGS):
        for class_id, class_name in enumerate(CLASSES):
            rows = [lookup[template.format(f=finding.lower())] for template in TEMPLATES[class_name]]
            prototypes[fid, class_id] = F.normalize(torch.stack(rows).mean(0), dim=0)
    text_payload = {'text_cache_id': TEXT_CACHE_ID, 'texts': all_texts,
                    'embeddings': embeddings, 'prototypes': prototypes}
    atomic_save(text_payload, TEXT_CACHE)

TEXT_TO_ID = {text: index for index, text in enumerate(text_payload['texts'])}
TEXT_EMBEDDINGS = text_payload['embeddings'].float()
PROTOTYPES = text_payload['prototypes'].float()
assert PROTOTYPES.shape == (18, 3, 512)
print('Text targets:', len(TEXT_TO_ID), '| prototypes:', tuple(PROTOTYPES.shape))

## 9. Stage prefix tensors to local disk for training

Reading ~180 GiB of prior/current prefix tensors per epoch directly from Drive is slow.
This optional copy uses Colab local disk for the current session; Drive remains the
resumable source of truth.

In [ ]:
PREFIX_READ_DIR = PREFIX_DIR
if STAGE_PREFIX_TO_LOCAL:
    LOCAL_PREFIX_DIR.mkdir(parents=True, exist_ok=True)
    for volume in tqdm(target_volumes, desc='Stage prefixes locally'):
        source = prefix_path(volume)
        destination = prefix_path(volume, LOCAL_PREFIX_DIR)
        if not destination.is_file() or destination.stat().st_size != source.stat().st_size:
            shutil.copyfile(source, destination)
    PREFIX_READ_DIR = LOCAL_PREFIX_DIR
print('Training prefix directory:', PREFIX_READ_DIR)

## 10. Construct the trainable visual tail and Difference Transformer

In [ ]:
# Free modules not needed during training, then move only the tail to GPU.
clip.cpu(); gc.collect(); torch.cuda.empty_cache()
FINAL_VISUAL_BLOCK = visual.enc_temporal_transformer.layers[-1]
FINAL_VISUAL_NORM = visual.enc_temporal_transformer.norm_out
for parameter in FINAL_VISUAL_BLOCK.parameters(): parameter.requires_grad_(True)
for parameter in FINAL_VISUAL_NORM.parameters(): parameter.requires_grad_(True)
visual.vq.eval()
for parameter in visual.vq.parameters(): parameter.requires_grad_(False)
for parameter in clip.to_visual_latent.parameters(): parameter.requires_grad_(False)

class TrainableVisualTail(nn.Module):
    def __init__(self, block, norm, vq, projection):
        super().__init__()
        self.block = block; self.norm = norm; self.vq = vq; self.projection = projection

    def train(self, mode=True):
        super().train(mode)
        self.vq.eval(); self.projection.eval()
        return self

    def forward(self, prefix):
        # prefix: [B, 576, 24, 512]
        batch, spatial, temporal, dim = prefix.shape
        assert (spatial, temporal, dim) == PREFIX_SHAPE
        height = width = 24
        tokens = prefix.reshape(batch * spatial, temporal, dim)
        tokens = apply_transformer_layer(
            self.block, tokens, (batch, temporal, height, width))
        tokens = self.norm(tokens)
        tokens = rearrange(tokens, '(b h w) t d -> b t h w d',
                           b=batch, h=height, w=width)
        flat = tokens.reshape(batch, temporal * height * width, dim)
        quantized, _, _ = self.vq(flat)
        quantized = quantized.reshape(batch, temporal, height, width, dim)
        image_features = quantized.mean(dim=1).reshape(batch, -1)
        embeddings = self.projection(image_features)
        return F.normalize(embeddings.float(), dim=-1)

class DifferenceTransformer(nn.Module):
    def __init__(self, n_findings=18, d_in=512, d_model=256):
        super().__init__()
        self.W = nn.Linear(d_in, d_model)
        self.role = nn.Parameter(torch.randn(2, d_model) * 0.02)
        self.e_diff = nn.Parameter(torch.randn(1, d_model) * 0.02)
        self.finding_emb = nn.Embedding(n_findings, d_model)
        layer = nn.TransformerEncoderLayer(
            d_model, 4, d_model * 4, dropout=0.1, batch_first=True, activation='gelu')
        self.encoder = nn.TransformerEncoder(layer, 2)
        self.head = nn.Linear(d_model, d_in)
        self.mag_head = nn.Linear(d_model, 1)
        self.logit_scale = nn.Parameter(torch.tensor(float(np.log(1 / 0.07))))

    def forward(self, prior, current, finding_id):
        batch = prior.shape[0]
        diff = self.e_diff.expand(batch, -1) + self.finding_emb(finding_id)
        sequence = torch.stack([
            diff, self.W(prior) + self.role[0], self.W(current) + self.role[1]], dim=1)
        hidden = self.encoder(sequence)[:, 0]
        return self.head(hidden), self.mag_head(hidden).squeeze(-1)

class PartialTemporalModel(nn.Module):
    def __init__(self, tail, difference):
        super().__init__(); self.tail = tail; self.difference = difference

TAIL = TrainableVisualTail(FINAL_VISUAL_BLOCK, FINAL_VISUAL_NORM,
                           visual.vq, clip.to_visual_latent)
MODEL = PartialTemporalModel(TAIL, DifferenceTransformer()).to(DEVICE)

image_parameters = [parameter for module in (MODEL.tail.block, MODEL.tail.norm)
                    for parameter in module.parameters() if parameter.requires_grad]
temporal_parameters = list(MODEL.difference.parameters())
optimizer = torch.optim.AdamW([
    {'params': image_parameters, 'lr': IMAGE_LR},
    {'params': temporal_parameters, 'lr': TEMPORAL_LR},
], weight_decay=WEIGHT_DECAY)

print('Trainable final visual params:', f'{sum(p.numel() for p in image_parameters):,}')
print('Trainable temporal params:', f'{sum(p.numel() for p in temporal_parameters):,}')
assert all(not parameter.requires_grad for parameter in MODEL.tail.projection.parameters())
assert all(not parameter.requires_grad for parameter in MODEL.tail.vq.parameters())

## 11. Pair loading, losses, evaluation, and resumable checkpoints

In [ ]:
def load_prefix(volume):
    payload = torch.load(prefix_path(volume, PREFIX_READ_DIR), map_location='cpu', weights_only=False)
    assert payload['prefix_id'] == PREFIX_ID
    return payload['prefix']

train_labels = [finding['y'] for row in records['train'] for finding in row['findings']]
counts = torch.bincount(torch.tensor(train_labels), minlength=3).float()
CLASS_WEIGHTS = (counts.sum() / counts.clamp(min=1)); CLASS_WEIGHTS /= CLASS_WEIGHTS.mean()
ce_loss = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS.to(DEVICE))
print('Class counts:', counts.tolist(), '| weights:', CLASS_WEIGHTS.tolist())

def pair_tensors(record):
    prefixes = torch.stack([load_prefix(record['prior_volume']), load_prefix(record['curr_volume'])])
    finding_ids = torch.tensor([row['fid'] for row in record['findings']], dtype=torch.long)
    labels = torch.tensor([row['y'] for row in record['findings']], dtype=torch.long)
    text_ids = torch.tensor([TEXT_TO_ID[row['text']] for row in record['findings']], dtype=torch.long)
    return prefixes, finding_ids, labels, text_ids

def forward_record(record):
    prefixes, finding_ids, labels, text_ids = pair_tensors(record)
    prefixes = prefixes.to(DEVICE, non_blocking=True)
    finding_ids = finding_ids.to(DEVICE); labels = labels.to(DEVICE)
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        image_embeddings = MODEL.tail(prefixes)
        prior = image_embeddings[0].unsqueeze(0).expand(len(labels), -1)
        current = image_embeddings[1].unsqueeze(0).expand(len(labels), -1)
        difference, magnitude = MODEL.difference(prior, current, finding_ids)
        normalized_difference = F.normalize(difference.float(), dim=-1)
        prototypes = PROTOTYPES[finding_ids.cpu()].to(DEVICE)
        logits = torch.einsum('bd,bkd->bk', normalized_difference,
                              F.normalize(prototypes, dim=-1)) * MODEL.difference.logit_scale.exp().clamp(max=100)
        target_text = TEXT_EMBEDDINGS[text_ids].to(DEVICE)
        loss_ce = ce_loss(logits, labels)
        loss_mag = F.binary_cross_entropy_with_logits(magnitude.float(), (labels != C2I['stable']).float())
        loss_text = (1.0 - (normalized_difference * F.normalize(target_text, dim=-1)).sum(-1)).mean()
        loss = LAMBDA_CE * loss_ce + LAMBDA_MAG * loss_mag + LAMBDA_TEXT * loss_text
    return loss, logits, labels, {'ce': loss_ce.item(), 'mag': loss_mag.item(), 'text': loss_text.item()}

@torch.inference_mode()
def evaluate(split):
    MODEL.eval(); all_true, all_pred = [], []; total_loss = 0.0; findings = 0
    for record in tqdm(records[split], desc=f'eval {split}', leave=False):
        loss, logits, labels, _ = forward_record(record)
        count = len(labels); total_loss += float(loss) * count; findings += count
        all_true.extend(labels.cpu().tolist()); all_pred.extend(logits.argmax(1).cpu().tolist())
    return {
        'loss': total_loss / max(findings, 1),
        'macro_f1': f1_score(all_true, all_pred, labels=[0,1,2], average='macro', zero_division=0),
        'balanced_accuracy': balanced_accuracy_score(all_true, all_pred),
        'per_class_f1': f1_score(all_true, all_pred, labels=[0,1,2], average=None, zero_division=0).tolist(),
        'n_findings': findings,
    }

BEST_PATH = CHECKPOINT_DIR / 'best.pt'
LAST_PATH = CHECKPOINT_DIR / 'last.pt'

def checkpoint_payload(epoch, best_metric, bad_epochs):
    return {
        'epoch': epoch, 'best_metric': best_metric, 'bad_epochs': bad_epochs,
        'final_visual_block': MODEL.tail.block.state_dict(),
        'final_visual_norm': MODEL.tail.norm.state_dict(),
        'difference': MODEL.difference.state_dict(),
        'optimizer': optimizer.state_dict(),
        'config': {'image_lr': IMAGE_LR, 'temporal_lr': TEMPORAL_LR,
                    'prefix_id': PREFIX_ID, 'ctclip_commit': CTCLIP_COMMIT, 'seed': SEED,
                    'cohort_id': COHORT_ID, 'label_sha256': LABEL_SHA256},
    }

def save_checkpoint(path, epoch, best_metric, bad_epochs):
    atomic_save(checkpoint_payload(epoch, best_metric, bad_epochs), path)

def load_checkpoint(path, load_optimizer=True):
    payload = torch.load(path, map_location='cpu', weights_only=False)
    assert payload['config']['prefix_id'] == PREFIX_ID
    assert payload['config']['ctclip_commit'] == CTCLIP_COMMIT
    assert payload['config']['cohort_id'] == COHORT_ID
    assert payload['config']['label_sha256'] == LABEL_SHA256
    MODEL.tail.block.load_state_dict(payload['final_visual_block'])
    MODEL.tail.norm.load_state_dict(payload['final_visual_norm'])
    MODEL.difference.load_state_dict(payload['difference'])
    if load_optimizer: optimizer.load_state_dict(payload['optimizer'])
    return payload

## 12. Benchmark 10 pairs before committing to the run

This performs forward/backward without optimizer updates, then estimates epoch duration.

In [ ]:
MODEL.train(); MODEL.tail.vq.eval(); optimizer.zero_grad(set_to_none=True)
benchmark_rows = records['train'][:10]
torch.cuda.synchronize(); start = time.time()
for record in tqdm(benchmark_rows, desc='benchmark'):
    loss, _, _, _ = forward_record(record)
    (loss / len(benchmark_rows)).backward()
torch.cuda.synchronize(); elapsed = time.time() - start
optimizer.zero_grad(set_to_none=True); gc.collect(); torch.cuda.empty_cache()
seconds_per_pair = elapsed / len(benchmark_rows)
print('Seconds/pair:', round(seconds_per_pair, 2))
print('Estimated train-only hours/epoch:', round(seconds_per_pair * len(records['train']) / 3600, 2))
print('Estimated 10-epoch train-only hours:', round(seconds_per_pair * len(records['train']) * 10 / 3600, 2))

## 13. Train with epoch-level resume and early stopping

Re-running resumes from `last.pt`. The held-out test set is not used for early stopping.

In [ ]:
start_epoch = 1; best_metric = -1.0; bad_epochs = 0
if LAST_PATH.is_file():
    resumed = load_checkpoint(LAST_PATH, load_optimizer=True)
    start_epoch = int(resumed['epoch']) + 1
    best_metric = float(resumed['best_metric']); bad_epochs = int(resumed['bad_epochs'])
    print('Resuming at epoch', start_epoch, '| best tune macro-F1', best_metric)

history_path = RESULT_DIR / 'history.json'
history = json.loads(history_path.read_text()) if history_path.is_file() else []
for epoch in range(start_epoch, EPOCHS + 1):
    MODEL.train(); MODEL.tail.vq.eval()
    order = list(range(len(records['train']))); random.Random(SEED + epoch).shuffle(order)
    optimizer.zero_grad(set_to_none=True)
    running = Counter(); seen_findings = 0
    for step, index in enumerate(tqdm(order, desc=f'epoch {epoch}')):
        loss, logits, labels, parts = forward_record(records['train'][index])
        (loss / GRAD_ACCUM_PAIRS).backward()
        count = len(labels); seen_findings += count
        running['loss'] += float(loss) * count
        for key, value in parts.items(): running[key] += value * count
        if (step + 1) % GRAD_ACCUM_PAIRS == 0 or step + 1 == len(order):
            nn.utils.clip_grad_norm_([p for p in MODEL.parameters() if p.requires_grad], MAX_GRAD_NORM)
            optimizer.step(); optimizer.zero_grad(set_to_none=True)

    tune = evaluate('tune')
    train_summary = {key: value / max(seen_findings, 1) for key, value in running.items()}
    improved = tune['macro_f1'] > best_metric
    if improved:
        best_metric = tune['macro_f1']; bad_epochs = 0
        save_checkpoint(BEST_PATH, epoch, best_metric, bad_epochs)
    else:
        bad_epochs += 1
    save_checkpoint(LAST_PATH, epoch, best_metric, bad_epochs)
    row = {'epoch': epoch, 'train': train_summary, 'tune': tune,
           'best_tune_macro_f1': best_metric, 'bad_epochs': bad_epochs}
    history.append(row); print(json.dumps(row, indent=2))
    history_path.write_text(json.dumps(history, indent=2) + '\n')
    if bad_epochs >= PATIENCE:
        print('Early stopping.'); break

## 14. Final held-out test evaluation

In [ ]:
assert BEST_PATH.is_file(), 'No best checkpoint exists.'
best = load_checkpoint(BEST_PATH, load_optimizer=False)
tune_metrics = evaluate('tune')
test_metrics = evaluate('test')
final_results = {
    'best_epoch': int(best['epoch']),
    'tune': tune_metrics,
    'test': test_metrics,
    'trainable_final_visual_params': sum(p.numel() for p in image_parameters),
    'trainable_temporal_params': sum(p.numel() for p in temporal_parameters),
    'cohort': {'id': COHORT_ID, 'train_pairs': 3172, 'tune_pairs': 555, 'test_pairs': 233,
               'unique_volumes': 6590, 'train_findings': 10603, 'tune_findings': 1882,
               'test_findings': 795},
}
(RESULT_DIR / 'final_results.json').write_text(json.dumps(final_results, indent=2) + '\n')
print(json.dumps(final_results, indent=2))

eval tune:   0%|          | 0/555 [00:00<?, ?it/s]

eval test:   0%|          | 0/233 [00:00<?, ?it/s]

{
  "best_epoch": 3,
  "tune": {
    "loss": 1.1709731701329968,
    "macro_f1": 0.5780196078513702,
    "balanced_accuracy": 0.5718888568152479,
    "per_class_f1": [
      0.7387218045112782,
      0.41732283464566927,
      0.5780141843971631
    ],
    "n_findings": 1882
  },
  "test": {
    "loss": 1.2102198614076998,
    "macro_f1": 0.5254796807386137,
    "balanced_accuracy": 0.521604513812306,
    "per_class_f1": [
      0.7076222980659841,
      0.3069306930693069,
      0.5618860510805501
    ],
    "n_findings": 795
  },
  "trainable_final_visual_params": 2637888,
  "trainable_temporal_params": 1848066,
  "cohort": {
    "id": "v6_synthetic_direction_report_explicit_grounded_v1",
    "train_pairs": 3172,
    "tune_pairs": 555,
    "test_pairs": 233,
    "unique_volumes": 6590,
    "train_findings": 10603,
    "tune_findings": 1882,
    "test_findings": 795
  }
}


## Interpretation

Compare `final_results.json` with a frozen CT-CLIP baseline using the same deterministic
v6 report-grounded 3,172/555/233 pair split. This experiment is **partially unfrozen
CT-CLIP**: only the final temporal CTViT block and its final norm adapt. The text tower and
CT-CLIP visual projection remain frozen, so the original 512-d text space stays fixed while
the last visual block learns task-specific temporal progression features.